# Part 7 · Notebook 08 — Hedging a portfolio

**Sessions:** S8 (Hedging strategies & M3b release) · [Lesson plan](../../docs/lessons/PART_07_STRATEGY_LIBRARY.md) · graded labs in [`labs/part07/`](../../labs/part07/)

**You will:**
1. Size an index-futures hedge with beta.
2. Find the call strike that makes a collar cost nothing.
3. Compare no hedge, puts, a collar and futures through a crash.
4. Judge a hedge by the portfolio's drawdown and its cost, not by its own P&L.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic, built from regimes you know, and every strategy here is a **hypothesis** with a first-look evaluation: the honest backtest comes in Part 8.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p7lib.py is in notebooks/part07/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p7lib as p

p.use_course_style()

## 1. A futures hedge

To hedge a portfolio with index futures, sell `round(hedge_ratio × β × value / (futures price × multiplier))` contracts. β is the portfolio's sensitivity to the index; the multiplier is 50 for ES, 5 for MES.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def futures_hedge_contracts(value, beta, fut_price, multiplier, hedge_ratio=1.0):
    return ...                                    # ✍️ an int

cases = [(2_000_000, 1.2, 5000.0, 50, 1.0), (2_000_000, 1.2, 5000.0, 5, 0.5), (350_000, 0.8, 5000.0, 5, 1.0)]
mine = [p.attempt(futures_hedge_contracts, *c) for c in cases]
mine = p.check("futures_hedge_contracts", mine, [p.futures_hedge_contracts(*c) for c in cases])
dict(zip(["$2m, β 1.2, full, ES", "$2m, β 1.2, half, MES", "$350k, β 0.8, full, MES"], mine))

A small account can't hedge precisely with ES (one contract is $250k of index); micro contracts make the rounding error tolerable.

## 2. A zero-cost collar

Buy a protective put at `k_put` and pay for it by selling a call: find the call strike **above spot** whose premium equals the put's. Price each strike with its own IV and solve with `brentq` on `[S, 3S]`.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
from scipy.optimize import brentq

def zero_cost_call_strike(S, T, k_put, iv_fn, r=0.04, q=0.0):
    put = p.bsm_price(S, k_put, T, r, q, iv_fn(k_put), -1)
    f = ...                                       # ✍️ a function of k: the call premium at k minus the put premium
    return float(brentq(f, S, 3 * S, xtol=1e-8))

S = 600.0
cases = [(S, 0.25, 0.90 * S, p.skew_iv(S)), (S, 0.25, 0.95 * S, p.skew_iv(S)), (S, 0.25, 0.90 * S, lambda k: 0.18)]
mine = [p.attempt(zero_cost_call_strike, *c) for c in cases]
mine = p.check("zero_cost_call_strike", mine, [p.zero_cost_call_strike(*c) for c in cases])
pd.DataFrame({"put strike": [c[2] for c in cases], "smile": ["skewed", "skewed", "flat 18%"], "zero-cost call strike": mine}).round(2)

With a skewed smile the put is expensive and the call cheap, so the call you must sell is much closer to spot: the skew is the price of protection.

## 3. Through a crash

One million in the index: two calm years, a 30% crash in three weeks, then a recovery. Four choices: no hedge; rolling 10% out-of-the-money 3-month puts every month; the same puts with zero-cost calls (a collar); and shorting half the position in futures.

In [ ]:
close, iv = p.crash_path()
curves = {m: p.hedge_study(close, iv, m) for m in ("none", "puts", "collar", "futures")}
fig, ax = plt.subplots(figsize=(11, 4))
for m, eq in curves.items():
    ax.plot(eq.index, eq / 1e6, label=m)
ax.set(ylabel="equity, $m", title="Hedges through a crash"); ax.legend(); plt.show()

Judge each hedge by the **portfolio**: its total return, its maximum drawdown, and the difference in total return versus no hedge (`cost_vs_none`, negative when the hedge cost more than it gave back). Compute the maximum drawdown of an equity curve: the most negative `equity / running maximum − 1`.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def hedge_report(curves):
    rows = {}
    for m, eq in curves.items():
        rows[m] = {"total_return": eq.iloc[-1] / eq.iloc[0] - 1,
                   "max_drawdown": ...}           # ✍️
    df = pd.DataFrame.from_dict(rows, orient="index")
    df["cost_vs_none"] = df["total_return"] - df.loc["none", "total_return"]
    return df

mine = p.attempt(hedge_report, curves)
mine = p.check("hedge_report (through the crash)", mine, p.hedge_report(curves))
calm = {m: eq.iloc[:500] for m, eq in curves.items()}
display(mine.round(3))
print("the calm years only (the insurance premium, with nothing to show for it):")
p.hedge_report(calm).round(3)

In the calm years the puts and the futures cost money (the collar roughly breaks even, having given up upside a calm market didn't deliver), and a hedge judged by its own P&L gets cancelled just before it is needed (common mistake #12). Through the crash each one cuts the drawdown, at different prices: puts keep the upside, the collar gives it up to finance the puts, and futures remove return in both directions.

## Wrap-up

* Size futures hedges with β and the right contract size.
* Collars trade upside for protection at a price set by the skew.
* Evaluate hedges at the portfolio level: drawdown reduction per unit of cost over a full cycle.
* Graded version: `labs/part07/week24_vol_hedging` (futures sizing, zero-cost collar, the hedging study).